# Model Training: Student Depression Prediction

This notebook trains and evaluates machine learning models to predict student depression based on various features.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_curve, auc,
                             roc_auc_score)
import joblib

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully!")

## 1. Load and Preprocess Data

In [ ]:
# Load the dataset
df = pd.read_csv('../data/student_depression_dataset.csv')

# Clean string columns - remove extra quotes from values
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip().str.strip("'")

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
# Drop unnecessary columns
columns_to_drop = ['id', 'City']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

# Handle missing values
print(f"Missing values before: {df.isnull().sum().sum()}")
df = df.dropna()
print(f"Missing values after: {df.isnull().sum().sum()}")
print(f"Final shape: {df.shape}")

## 2. Feature Encoding

In [ ]:
# Store encoders for later use
encoders = {}

# Target variable: Depression
le_target = LabelEncoder()
df['Depression'] = le_target.fit_transform(df['Depression'])
encoders['target'] = le_target
print(f"Target classes: {le_target.classes_}")

# Gender encoding
if 'Gender' in df.columns:
    le_gender = LabelEncoder()
    df['Gender'] = le_gender.fit_transform(df['Gender'])
    encoders['gender'] = le_gender
    print(f"Gender classes: {le_gender.classes_}")

# Ordinal encoding for Sleep Duration
if 'Sleep Duration' in df.columns:
    df['Sleep Duration'] = df['Sleep Duration'].replace('Others', '5-6 hours')
    sleep_order = ['Less than 5 hours', '5-6 hours', '7-8 hours', 'More than 8 hours']
    sleep_encoder = OrdinalEncoder(categories=[sleep_order], handle_unknown='use_encoded_value', unknown_value=-1)
    df['Sleep Duration'] = sleep_encoder.fit_transform(df[['Sleep Duration']])
    encoders['sleep'] = sleep_encoder
    print(f"Sleep Duration encoded ordinally")

# Ordinal encoding for Dietary Habits
if 'Dietary Habits' in df.columns:
    df['Dietary Habits'] = df['Dietary Habits'].replace('Others', 'Moderate')
    diet_order = ['Unhealthy', 'Moderate', 'Healthy']
    diet_encoder = OrdinalEncoder(categories=[diet_order], handle_unknown='use_encoded_value', unknown_value=-1)
    df['Dietary Habits'] = diet_encoder.fit_transform(df[['Dietary Habits']])
    encoders['diet'] = diet_encoder
    print(f"Dietary Habits encoded ordinally")

# Encode remaining categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"{col} encoded with {len(le.classes_)} classes")

print(f"\n✓ All features encoded successfully!")

## 3. Train-Test Split and Feature Scaling

In [ ]:
# Separate features and target
X = df.drop('Depression', axis=1)
y = df['Depression']

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Data split and scaled successfully!")

## 4. Model Training and Comparison

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42)
}

# Train and evaluate each model
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else None
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })
    
    print(f"  Accuracy: {accuracy:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

print("\n✓ All models trained!")

In [ ]:
# Display results comparison
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Recall', ascending=False)
results_df = results_df.round(4)

print("=" * 80)
print("MODEL COMPARISON RESULTS")
print("=" * 80)
display(results_df)

# Visualization of model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart for all metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(results_df))
width = 0.2

for i, metric in enumerate(metrics):
    axes[0].bar(x + i*width, results_df[metric], width, label=metric)

axes[0].set_xlabel('Models')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(results_df['Model'], rotation=45, ha='right')
axes[0].legend()
axes[0].set_ylim(0, 1)

# ROC-AUC comparison
axes[1].barh(results_df['Model'], results_df['ROC-AUC'], color='steelblue')
axes[1].set_xlabel('ROC-AUC Score')
axes[1].set_title('ROC-AUC Comparison')
for i, v in enumerate(results_df['ROC-AUC']):
    axes[1].text(v + 0.01, i, f'{v:.4f}', va='center')

plt.tight_layout()
plt.show()

## 5. Best Model: Random Forest Deep Dive

In [ ]:
# Train optimized Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',  # Handle class imbalance
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# Detailed classification report
print("=" * 60)
print("RANDOM FOREST - DETAILED PERFORMANCE")
print("=" * 60)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['No Depression', 'Depression']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Depression', 'Depression'],
            yticklabels=['No Depression', 'Depression'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_rf)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curve')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print("-" * 40)
for i, row in feature_importance.head(10).iterrows():
    print(f"{row['feature']}: {row['importance']:.4f}")

# Visualization
plt.figure(figsize=(10, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(feature_importance)))
plt.barh(feature_importance['feature'], feature_importance['importance'], color=colors)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Cross-Validation

In [ ]:
# 5-Fold Cross-Validation for Random Forest
print("Performing 5-Fold Cross-Validation...")
print("=" * 50)

# Different scoring metrics
scoring_metrics = ['accuracy', 'recall', 'precision', 'f1', 'roc_auc']

cv_results = {}
for metric in scoring_metrics:
    scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring=metric)
    cv_results[metric] = {
        'mean': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }
    print(f"{metric.upper():12s}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
cv_means = [cv_results[m]['mean'] for m in scoring_metrics]
cv_stds = [cv_results[m]['std'] for m in scoring_metrics]

bars = ax.bar(scoring_metrics, cv_means, yerr=cv_stds, capsize=5, color='steelblue', alpha=0.8)
ax.set_ylabel('Score')
ax.set_title('5-Fold Cross-Validation Results')
ax.set_ylim(0, 1)

for bar, mean in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{mean:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning (Optional)

In [ ]:
# GridSearchCV for hyperparameter tuning (uncomment to run - takes a few minutes)
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [None, 10, 20, 30],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4]
# }

# grid_search = GridSearchCV(
#     RandomForestClassifier(class_weight='balanced', random_state=42),
#     param_grid,
#     cv=5,
#     scoring='recall',
#     n_jobs=-1,
#     verbose=1
# )

# grid_search.fit(X_train_scaled, y_train)
# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best recall score: {grid_search.best_score_:.4f}")

print("⏭ Hyperparameter tuning skipped (uncomment code to run)")
print("  Current model uses default optimized parameters")

## 9. Save Models

In [ ]:
# Create models directory if it doesn't exist
import os
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Save the Random Forest model
joblib.dump(rf_model, os.path.join(models_dir, 'random_forest_model.pkl'))
print("✓ Model saved to models/random_forest_model.pkl")

# Save the scaler
joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))
print("✓ Scaler saved to models/scaler.pkl")

# Save the encoders
joblib.dump(encoders, os.path.join(models_dir, 'encoders.pkl'))
print("✓ Encoders saved to models/encoders.pkl")

# Save feature importance
feature_importance.to_csv(os.path.join(models_dir, 'feature_importance.csv'), index=False)
print("✓ Feature importance saved to models/feature_importance.csv")

# Save feature names
feature_names = X.columns.tolist()
joblib.dump(feature_names, os.path.join(models_dir, 'feature_names.pkl'))
print("✓ Feature names saved to models/feature_names.pkl")

## 10. Model Summary

In [ ]:
# Final Summary
print("=" * 70)
print("MODEL TRAINING COMPLETE!")
print("=" * 70)

print(f"\n📊 DATASET:")
print(f"   • Total samples: {len(df):,}")
print(f"   • Training samples: {len(X_train):,}")
print(f"   • Test samples: {len(X_test):,}")
print(f"   • Features: {len(X.columns)}")

print(f"\n🏆 BEST MODEL: Random Forest")
print(f"   • Accuracy: {accuracy_score(y_test, y_pred_rf):.2%}")
print(f"   • Recall: {recall_score(y_test, y_pred_rf):.2%}")
print(f"   • Precision: {precision_score(y_test, y_pred_rf):.2%}")
print(f"   • F1-Score: {f1_score(y_test, y_pred_rf):.2%}")
print(f"   • ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.2%}")

print(f"\n🔑 TOP 5 IMPORTANT FEATURES:")
for i, row in feature_importance.head(5).iterrows():
    print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")

print(f"\n💾 SAVED ARTIFACTS:")
print("   • models/random_forest_model.pkl")
print("   • models/scaler.pkl")
print("   • models/encoders.pkl")
print("   • models/feature_importance.csv")
print("   • models/feature_names.pkl")

print("\n" + "=" * 70)
print("🚀 Ready to deploy! Run: streamlit run app.py")
print("=" * 70)